# Random Forest — baseline trên 63 landmark

Bộ 6 lớp: `fist/one/peace/three/four/palm → 0/1/2/3/4/5`.
Gắn dataset **thanhmay2406/hand-gesture** bằng Add Input. Notebook tự tìm dữ liệu;
đặt `DATA_ROOT` nếu có nhiều bộ phù hợp. Output nằm trong `/kaggle/working/hand_gesture/`.

Giữ split đã cố định **27.080 / 5.803 / 5.804**, seed 42. Không chia lại, không cân bằng lại lớp.
Đánh giá các mẫu có cả ảnh và landmark hợp lệ; split chưa tách người chụp, công thức
chuẩn hóa landmark/crop chưa được xác minh. Accuracy này không đại diện cho webcam end-to-end.
Không cam kết baseline đạt 92%. `SMOKE_TEST=True` chỉ kiểm tra code với tập nhỏ, không dùng làm kết quả thực nghiệm.

In [ ]:
from pathlib import Path
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
DATA_ROOT = None  # ví dụ: '/kaggle/input/hand-gesture'
OUTPUT_ROOT = Path('/kaggle/working/hand_gesture')
SEED = 42
SMOKE_TEST = False
SMOKE_PER_CLASS = {'train': 8, 'validation': 4, 'test': 4}
DEVICE = 'auto'  # auto/cpu/cuda; RF luôn chạy CPU
NUM_WORKERS = 2
BATCH_SIZE = 256
MAX_EPOCHS = 1
PATIENCE = 8
RUN_NAME = None  # Tên phiên trong metadata; thư mục output luôn theo MODEL_NAME

MODEL_NAME = 'rf'

### Kiểm tra dữ liệu
Hash manifest/ảnh, schema, mapping và split phải hợp lệ; lỗi sẽ dừng notebook.

In [ ]:
from pathlib import Path
import copy
import hashlib
import importlib.metadata
import json
import os
import platform
import random
import time

import joblib
import numpy as np
import pandas as pd
import matplotlib
from IPython import get_ipython
if get_ipython() is None:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageEnhance
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)
from IPython.display import display, Markdown

GESTURES = ['fist', 'one', 'peace', 'three', 'four', 'palm']
CLASS_MAPPING = [{'class_id':i, 'gesture':name} for i,name in enumerate(GESTURES)]
LM_COLUMNS = [f'lm_{i}' for i in range(63)]
MODEL_ARTIFACTS = {'rf':['model.joblib','model_config.json','class_mapping.json'],
    'mlp':['model.pt','scaler.joblib','model_config.json','class_mapping.json'],
    'cnn':['model.pt','model_config.json','class_mapping.json']}
EXPECTED_SPLITS = {'train':27080, 'validation':5803, 'test':5804}
MODEL_ORDER = ['rf','mlp','cnn']
METRIC_COLUMNS = ['accuracy','macro_precision','macro_recall','macro_f1']
HISTORY_COLUMNS = ['step','epoch','learning_rate','train_loss','validation_loss',
                   'train_accuracy','validation_accuracy','train_macro_f1','validation_macro_f1','smoke_test']
PLOT_STYLE = {'font.family':'DejaVu Sans','font.size':10,'axes.titlesize':12,
              'axes.prop_cycle':plt.cycler(color=['#1675cb','#ed8733','#37a36b'])}


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()


def find_data_root(explicit=None, search_root='/kaggle/input'):
    if explicit is not None:
        root = Path(explicit).expanduser().resolve()
        if not (root/'metadata/manifest.csv').is_file() or not (root/'images').is_dir():
            raise ValueError(f'DATA_ROOT không chứa metadata/manifest.csv và images/: {root}')
        return root
    candidates = sorted({p.parent.parent for p in Path(search_root).rglob('manifest.csv')
                         if p.parent.name == 'metadata' and (p.parent.parent/'images').is_dir()})
    if len(candidates) != 1:
        raise ValueError(f'Tìm thấy {len(candidates)} bộ dữ liệu. Đặt DATA_ROOT thủ công. Candidates: {candidates}')
    return candidates[0]


def validate_image(root, row):
    relative = Path(row.image_path)
    pieces = row.sample_id.split('/', 1)
    if len(pieces) != 2:
        raise ValueError(f'ID mẫu sai: {row.sample_id}')
    expected = f'images/{row.split}/{row.gesture}/{pieces[1]}'
    if relative.is_absolute() or '..' in relative.parts or row.image_path != expected:
        raise ValueError(f'Đường dẫn ảnh sai: {row.image_path}')
    path = Path(root)/relative
    if not path.is_file():
        raise FileNotFoundError(path)
    if file_sha256(path) != row.image_sha256:
        raise ValueError(f'SHA-256 ảnh không khớp manifest: {row.image_path}')


def load_dataset(root):
    root = Path(root)
    report = json.loads((root/'metadata/preparation_report.json').read_text())
    manifest_path = root/'metadata/manifest.csv'
    digest = file_sha256(manifest_path)
    if digest != report['artifact_sha256']['manifest.csv']:
        raise ValueError('Hash manifest không khớp preparation_report.json.')
    frame = pd.read_csv(manifest_path, float_precision='round_trip')
    required = ['sample_id','source_label','class_id','gesture','image_path','image_sha256','split',*LM_COLUMNS]
    if set(required)-set(frame.columns) or {col for col in frame.columns if col.startswith('lm_')} != set(LM_COLUMNS):
        raise ValueError('Manifest thiếu cột bắt buộc.')
    if frame[required].isna().any().any() or not np.isfinite(frame[LM_COLUMNS].to_numpy(dtype=float)).all():
        raise ValueError('Manifest có dữ liệu thiếu/không hữu hạn.')
    if not frame.sample_id.is_unique or not frame.image_sha256.is_unique or not frame.image_path.is_unique:
        raise ValueError('ID/hash/path ảnh trùng, có thể rò rỉ giữa các split.')
    if (set(frame['split']) != set(EXPECTED_SPLITS)
            or frame['split'].value_counts().to_dict() != report['split_counts']):
        raise ValueError('Split không khớp preparation_report.json.')
    for mapping in CLASS_MAPPING:
        class_id, gesture = mapping['class_id'], mapping['gesture']
        rows = frame.loc[frame.class_id.eq(class_id)]
        if rows.empty or not rows.gesture.eq(gesture).all() or not rows.source_label.eq('train_val_'+gesture).all():
            raise ValueError('Mapping lớp trong manifest không khớp 6 lớp đã chốt.')
        source = report['class_mapping'].get('train_val_'+gesture, {})
        if source.get('class_id') != class_id or source.get('gesture') != gesture:
            raise ValueError('Mapping trong báo cáo không khớp.')
    if set(frame.class_id) != set(range(6)):
        raise ValueError('Manifest có class ID ngoài 0..5.')
    for split in EXPECTED_SPLITS:
        if set(frame.loc[frame.split.eq(split),'class_id']) != set(range(6)):
            raise ValueError(f'Split {split} thiếu lớp.')
    for row in frame[['sample_id','split','gesture','image_path','image_sha256']].itertuples(index=False):
        validate_image(root, row)
    return frame.sort_values('sample_id').reset_index(drop=True), report


def smoke_subset(frame, per_class, seed=42):
    parts=[]
    for split in ['train','validation','test']:
        for class_id in range(6):
            rows=frame.loc[frame.split.eq(split)&frame.class_id.eq(class_id)].sort_values('sample_id')
            if len(rows)<per_class[split]:
                raise ValueError('Không đủ mẫu smoke test theo lớp/split.')
            parts.append(rows.sample(per_class[split], random_state=seed+class_id))
    return pd.concat(parts).sort_values('sample_id').reset_index(drop=True)


def fit_scaler(train_features):
    scaler=StandardScaler()
    transformed=scaler.fit_transform(train_features).astype(np.float32)
    return scaler, transformed


def classification_metrics(y_true, probabilities):
    predictions=np.asarray(probabilities).argmax(axis=1)
    precision,recall,f1,_=precision_recall_fscore_support(
        y_true,predictions,labels=range(6),average='macro',zero_division=0)
    return {'accuracy':float(accuracy_score(y_true,predictions)), 'macro_precision':float(precision),
            'macro_recall':float(recall),'macro_f1':float(f1),
            'classification_report':classification_report(y_true,predictions,labels=range(6),
                target_names=GESTURES,output_dict=True,zero_division=0)}


def save_json(path, value):
    Path(path).write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False)+'\n')


def environment_info(device='cpu'):
    cpu_name = platform.processor()
    cpu_file = Path('/proc/cpuinfo')
    if cpu_file.is_file():
        cpu_name = next((line.split(':',1)[1].strip() for line in cpu_file.read_text().splitlines() if line.startswith('model name')), 'unknown')
    info={'python':platform.python_version(),'platform':platform.platform(),'cpu':cpu_name,
          'cpu_count':os.cpu_count(),'device':str(device),
          'versions':{p:importlib.metadata.version(p) for p in ['numpy','pandas','scikit-learn','Pillow','joblib','matplotlib']}}
    return info


def protocol_info(frame, source_manifest, smoke, seed):
    return {'manifest_sha256':source_manifest,'class_mapping':CLASS_MAPPING,
            'smoke_test':bool(smoke),'seed':seed,
            'sample_counts':frame['split'].value_counts().to_dict(),
            'test_sample_ids':frame.loc[frame.split.eq('test'),'sample_id'].tolist(),
            'limits':['random split does not separate subjects','crop/landmark provenance unverified',
                      'paired classifier evaluation, not webcam end-to-end']}


def standardize_history(history, model_name, config, smoke=False):
    """Retain observed values; leave unavailable/non-applicable measurements blank."""
    history=history.copy()
    if 'step' not in history:history['step']=np.arange(1,len(history)+1)
    if model_name=='rf':
        for column in ['epoch','learning_rate','train_loss','validation_loss']:history[column]=np.nan
    elif 'learning_rate' not in history and config.get('scheduler') is None:
        history['learning_rate']=config.get('learning_rate',np.nan)
    if 'smoke_test' not in history:history['smoke_test']=bool(smoke)
    return history.reindex(columns=HISTORY_COLUMNS)


def evaluation_tables(metrics):
    summary=[];per_class=[]
    for split,values in [('validation',metrics['validation_metrics']),('test',metrics)]:
        summary.append({'model':metrics['model'],'split':split,
                        'sample_count':metrics['protocol']['sample_counts'][split],
                        **{key:values[key] for key in METRIC_COLUMNS}})
        for class_id,gesture in enumerate(GESTURES):
            row=values['classification_report'][gesture]
            per_class.append({'model':metrics['model'],'split':split,'class_id':class_id,'gesture':gesture,
                              'precision':row['precision'],'recall':row['recall'],
                              'f1_score':row['f1-score'],'support':int(row['support'])})
    return pd.DataFrame(summary),pd.DataFrame(per_class)


def validate_result_metrics(metrics,predictions):
    """Check stored test measurements against predictions without replacing them."""
    measured=classification_metrics(predictions.y_true,predictions[[f'prob_{i}' for i in range(6)]].to_numpy())
    for key in METRIC_COLUMNS:
        if not np.isclose(metrics[key],measured[key],rtol=1e-9,atol=1e-12):
            raise ValueError(f"{metrics['model']}: {key} khác CSV dự đoán.")
    for gesture in GESTURES:
        for key in ['precision','recall','f1-score','support']:
            if not np.isclose(metrics['classification_report'][gesture][key],
                              measured['classification_report'][gesture][key],rtol=1e-9,atol=1e-12):
                raise ValueError(f"{metrics['model']}: {gesture}/{key} khác CSV dự đoán.")
    for split,values in [('validation',metrics['validation_metrics']),('test',metrics)]:
        support=sum(values['classification_report'][g]['support'] for g in GESTURES)
        if support!=metrics['protocol']['sample_counts'][split]:
            raise ValueError(f"{metrics['model']}: support {split} không khớp protocol.")


def finish_figure(fig,path=None):
    # Fixed canvas: no tight bounding-box cropping, so matching chart types have identical sizes.
    if path:fig.savefig(path,dpi=130)
    if get_ipython() is not None and plt.get_backend().lower()!='agg':plt.show()
    plt.close(fig)


def show_history(history,path,smoke=False,model_name=None):
    with plt.rc_context(PLOT_STYLE):
        fig,axes=plt.subplots(1,4,figsize=(18,4),layout='constrained')
        if smoke:fig.suptitle('SMOKE TEST — chỉ kiểm tra pipeline')
        if model_name=='rf':
            for index,label in [(0,'Loss'),(3,'Learning rate')]:
                axes[index].axis('off');axes[index].text(.5,.5,label+'\nKhông áp dụng cho RF',ha='center',va='center')
            for ax,metric,label in zip(axes[1:3],['accuracy','macro_f1'],['Accuracy','Macro-F1']):
                ax.bar(['train','validation'],[history[f'{s}_{metric}'].iloc[0] for s in ['train','validation']],
                       color=['#1675cb','#ed8733'])
                ax.set(title=label,ylim=(0,1),xlabel='Một lượt fit')
        else:
            for ax,metric,label in zip(axes,['loss','accuracy','macro_f1'],['Loss','Accuracy','Macro-F1']):
                for split in ['train','validation']:
                    ax.plot(history.epoch,history[f'{split}_{metric}'],label=split)
                ax.set(title=label,xlabel='Epoch');ax.legend()
                if metric!='loss':ax.set_ylim(0,1)
            if 'learning_rate' in history and history.learning_rate.notna().any():
                axes[3].plot(history.epoch,history.learning_rate)
                axes[3].set(title='Learning rate',xlabel='Epoch',yscale='log')
            else:
                axes[3].axis('off');axes[3].text(.5,.5,'Learning rate chưa được ghi',ha='center',va='center')
        finish_figure(fig,path)


def markdown_table(frame):
    """Small dependency-free Markdown tables, with blanks rendered as N/A."""
    def cell(value):
        if pd.isna(value):return 'N/A'
        if isinstance(value,(float,np.floating)):return f'{value:.6f}'
        return str(value).replace('|','\\|').replace('\n',' ')
    lines=['| '+' | '.join(frame.columns)+' |','| '+' | '.join(['---']*len(frame.columns))+' |']
    lines.extend('| '+' | '.join(cell(v) for v in row)+' |' for row in frame.itertuples(index=False,name=None))
    return '\n'.join(lines)


def top_confusions(predictions,model_name,limit=3):
    cm=confusion_matrix(predictions.y_true,predictions.y_pred,labels=range(6))
    errors=sorted([(int(cm[i,j]),i,j) for i in range(6) for j in range(6) if i!=j and cm[i,j]>0],
                  key=lambda x:(-x[0],x[1],x[2]))[:limit]
    return [{'model':model_name,'true':GESTURES[i],'predicted':GESTURES[j],'count':count} for count,i,j in errors]


def write_model_report(output_dir,metrics,predictions):
    output_dir=Path(output_dir);name=metrics['model'];config=metrics['config'];env=metrics['environment']
    summary,_=evaluation_tables(metrics)
    confusions=pd.DataFrame(top_confusions(predictions,name),columns=['model','true','predicted','count'])
    lines=[f'# Kết quả {name.upper()}','']
    if metrics['protocol']['smoke_test']:lines+=['**SMOKE TEST — chỉ kiểm tra pipeline, không dùng để chọn model.**','']
    lines+=['## Cấu hình','',f"Phiên: `{config.get('run_name',name)}`; seed: {metrics['protocol']['seed']}.",
            f"Kiến trúc: {config.get('architecture','Random Forest')}. ",
            f"Preprocessing: {config.get('preprocessing','Không có thông tin')}."]
    if name=='cnn' and config.get('run_name')=='cnn_tuned':lines+=['CNN trong báo cáo là phiên tinh chỉnh `cnn_tuned`.']
    if name=='rf':
        lines += [f"Số cây: {config.get('n_estimators','N/A')}; max depth: {config.get('max_depth','N/A')}. "]
    else:
        lines += [f"Optimizer: {config.get('optimizer','N/A')}; learning rate ban đầu: {config.get('learning_rate','N/A')}; "
                  f"weight decay: {config.get('weight_decay','N/A')}; batch size: {config.get('batch_size','N/A')}.",
                  f"Epoch tối đa: {config.get('max_epochs','N/A')}; patience: {config.get('patience','N/A')}; "
                  f"checkpoint được chọn tại epoch: {config.get('best_epoch','N/A')}.",
                  'Scheduler: `'+json.dumps(config.get('scheduler'),ensure_ascii=False)+'`.']
    lines+=['','[Cấu hình đầy đủ](model_config.json)','', '## Kết quả','',markdown_table(summary),'',
            '[Metric từng lớp](per_class_metrics.csv) · [Metric gốc](metrics.json) · [Dự đoán test](test_predictions.csv)',
            '', '## Phân tích nhầm lẫn','',markdown_table(confusions) if len(confusions) else 'Không có dự đoán sai.',
            '', '## Thời gian và thiết bị','',
            f"Training: {metrics['training_time_s']:.3f} s; fit wall time: {metrics['fit_wall_time_s']:.3f} s.",
            f"Device: {env.get('device')}; CPU: {env.get('cpu')}; GPU: {env.get('gpu') or 'N/A'}.",
            'RF đo estimator.fit; mạng đo train batches gồm loader, không gồm validation. '
            'Fit wall time của mạng gồm validation và chọn checkpoint. '
            'Không so sánh tốc độ train công bằng khi phần cứng khác nhau.',
            '', '## Biểu đồ','']
    for filename,label in [('history.png','Lịch sử huấn luyện'),('confusion_matrix.png','Confusion matrix'),
                           ('misclassified_gallery.png','Gallery dự đoán sai')]:
        if (output_dir/filename).is_file():lines.append(f'[{label}]({filename})')
    lines+=['','[Dữ liệu lịch sử](history.csv)','',
            'RF chỉ có một lượt fit; epoch/loss/learning rate không áp dụng. '
            'Train metrics của MLP/CNN được đo trong batch có dropout/augmentation; validation/test ở eval. '
            'Ô trống là số đo không áp dụng hoặc chưa có, không phải 0. '
            'Test được đánh giá sau khi model/checkpoint cố định. '
            'Split chưa tách người chụp; kết quả không đại diện FPS hoặc webcam end-to-end.']
    (output_dir/'report.md').write_text('\n'.join(lines)+'\n')


def write_model_outputs(output_dir,metrics,predictions):
    """Derive comparable tables and plots; never rewrite measured metrics or model artifacts."""
    output_dir=Path(output_dir)
    validate_result_metrics(metrics,predictions)
    summary,per_class=evaluation_tables(metrics)
    history=standardize_history(pd.read_csv(output_dir/'history.csv',float_precision='round_trip'),
                                metrics['model'],metrics['config'],metrics['protocol']['smoke_test'])
    history.to_csv(output_dir/'history.csv',index=False)
    summary.to_csv(output_dir/'summary.csv',index=False)
    per_class.to_csv(output_dir/'per_class_metrics.csv',index=False)
    cm=confusion_matrix(predictions.y_true,predictions.y_pred,labels=range(6))
    pd.DataFrame(cm,index=GESTURES,columns=GESTURES).to_csv(output_dir/'confusion_matrix.csv')
    pd.DataFrame(cm/np.maximum(cm.sum(axis=1,keepdims=True),1),index=GESTURES,columns=GESTURES).to_csv(output_dir/'confusion_matrix_normalized.csv')
    show_confusion(cm,output_dir/'confusion_matrix.png',smoke=metrics['protocol']['smoke_test'])
    show_history(history,output_dir/'history.png',smoke=metrics['protocol']['smoke_test'],model_name=metrics['model'])
    write_model_report(output_dir,metrics,predictions)


def export_evaluation(output_dir, model_name, frame, probabilities, protocol, env,
                      training_time, fit_wall_time, validation_metrics, config):
    output_dir=Path(output_dir);output_dir.mkdir(parents=True,exist_ok=True)
    metrics=classification_metrics(frame.class_id.to_numpy(),probabilities)
    artifact_names=MODEL_ARTIFACTS[model_name]
    metrics.update({'model':model_name,'protocol':protocol,'environment':env,
                    'training_time_s':float(training_time),'fit_wall_time_s':float(fit_wall_time),
                    'training_time_scope':'RF estimator.fit; neural training batches including loader, excluding validation',
                    'validation_metrics':validation_metrics,'config':config,
                    'artifact_sha256':{name:file_sha256(output_dir/name) for name in artifact_names},
                    'artifact_bytes':sum((output_dir/name).stat().st_size for name in artifact_names if (output_dir/name).is_file()),
                    'target_accuracy':.92,'target_met':metrics['accuracy']>=.92})
    result=frame[['sample_id','class_id','image_path']].rename(columns={'class_id':'y_true'}).copy()
    result['smoke_test']=protocol['smoke_test']
    result['y_pred']=np.asarray(probabilities).argmax(axis=1)
    for class_id in range(6):result[f'prob_{class_id}']=probabilities[:,class_id]
    result.to_csv(output_dir/'test_predictions.csv',index=False)
    metrics['test_predictions_sha256']=file_sha256(output_dir/'test_predictions.csv')
    save_json(output_dir/'metrics.json',metrics)
    write_model_outputs(output_dir,metrics,result)
    summary,per_class=evaluation_tables(metrics)
    display(summary.round(4))
    display(per_class.loc[per_class.split.eq('test')].round(4))
    display(Markdown(f"**{'SMOKE TEST — chưa phải huấn luyện đầy đủ. ' if protocol['smoke_test'] else ''}"
                     f"Test Accuracy: {metrics['accuracy']:.2%}; macro-F1: {metrics['macro_f1']:.4f}. "
                     f"{'Đạt' if metrics['target_met'] else 'Chưa đạt'} mục tiêu 92%.**"))
    return metrics,result


def show_confusion(cm, path=None, smoke=False):
    with plt.rc_context(PLOT_STYLE):
        return _show_confusion(cm,path,smoke)


def _show_confusion(cm, path=None, smoke=False):
    fig,axes=plt.subplots(1,2,figsize=(12,5),layout='constrained')
    if smoke:fig.suptitle('SMOKE TEST — chỉ kiểm tra pipeline')
    normalized=cm/np.maximum(cm.sum(axis=1,keepdims=True),1)
    for ax,values,title in zip(axes,[cm,normalized],['Số lượng','Tỷ lệ theo nhãn thật']):
        vmax=1 if values is normalized else max(int(cm.sum(axis=1).max()),1)
        im=ax.imshow(values,cmap='Blues',vmin=0,vmax=vmax)
        ax.set_xticks(range(6),GESTURES,rotation=35,ha='right');ax.set_yticks(range(6),GESTURES)
        ax.set(title=title,xlabel='Dự đoán',ylabel='Nhãn thật')
        for i in range(6):
            for j in range(6):
                label=f'{values[i,j]:.2f}' if values is normalized else str(int(values[i,j]))
                ax.text(j,i,label,ha='center',va='center',color='white' if values[i,j]>vmax/2 else 'black')
        fig.colorbar(im,ax=ax,shrink=.8)
    finish_figure(fig,path)


def show_errors(predictions,data_root,path,limit=12,smoke=False):
    with plt.rc_context(PLOT_STYLE):
        return _show_errors(predictions,data_root,path,limit,smoke)


def _show_errors(predictions,data_root,path,limit=12,smoke=False):
    errors=(predictions.loc[predictions.y_true.ne(predictions.y_pred)].sort_values('sample_id')
            .groupby('y_true',group_keys=False).head(2).head(limit))
    if errors.empty:
        Path(path).unlink(missing_ok=True)
        print('Không có dự đoán sai trong tập được đánh giá.');return
    rows=max(3,int(np.ceil(limit/4)))
    fig,axes=plt.subplots(rows,4,figsize=(12,3*rows),squeeze=False,layout='constrained')
    if smoke:fig.suptitle('SMOKE TEST — chỉ kiểm tra pipeline')
    for ax in axes.flat:ax.axis('off')
    for ax,row in zip(axes.flat,errors.itertuples()):
        with Image.open(Path(data_root)/row.image_path) as image:
            ax.imshow(ImageOps.exif_transpose(image).convert('RGB'))
        ax.set_title(f'Thật: {GESTURES[row.y_true]}\nDự đoán: {GESTURES[row.y_pred]}',fontsize=10)
    finish_figure(fig,path)

In [ ]:
root = find_data_root(DATA_ROOT)
full_frame, preparation_report = load_dataset(root)
if SEED != 42 or SEED != preparation_report['config']['seed']:
    raise ValueError('Baseline dùng seed 42, phải khớp báo cáo dữ liệu.')
frame = smoke_subset(full_frame, SMOKE_PER_CLASS, SEED) if SMOKE_TEST else full_frame
splits = {name: frame.loc[frame.split.eq(name)].reset_index(drop=True) for name in EXPECTED_SPLITS}
protocol = protocol_info(frame, file_sha256(root/'metadata/manifest.csv'), SMOKE_TEST, SEED)
output_dir = OUTPUT_ROOT / MODEL_NAME
output_dir.mkdir(parents=True, exist_ok=True)
save_json(output_dir/'class_mapping.json', CLASS_MAPPING)
display(frame.groupby(['split','gesture']).size().unstack().reindex(columns=GESTURES))
print('Manifest SHA-256:', protocol['manifest_sha256'])
print('SMOKE TEST' if SMOKE_TEST else 'HUẤN LUYỆN ĐẦY ĐỦ', '— output:', output_dir)

### Huấn luyện RF
300 cây, độ sâu tối đa 20, lá tối thiểu 2 mẫu. So sánh Accuracy train/validation để quan sát overfitting. Cấu hình này được chọn trước khi xem test.

In [ ]:
random.seed(SEED); np.random.seed(SEED)
env = environment_info('cpu')
display(env)
model_config = {'model':'rf','seed':SEED,'smoke_test':SMOKE_TEST,
    'n_estimators':10 if SMOKE_TEST else 300, 'max_depth':20,
    'min_samples_leaf':2, 'max_features':'sqrt','n_jobs':-1,
    'preprocessing':'63 landmark hiện có, không chuẩn hóa lại','features':LM_COLUMNS,
    'class_mapping':CLASS_MAPPING}
model = RandomForestClassifier(**{key:model_config[key] for key in
    ['n_estimators','max_depth','min_samples_leaf','max_features','n_jobs']}, random_state=SEED)
start = time.perf_counter()
model.fit(splits['train'][LM_COLUMNS].to_numpy(), splits['train'].class_id.to_numpy())
training_time = fit_wall_time = time.perf_counter()-start
train_metrics = classification_metrics(splits['train'].class_id, model.predict_proba(splits['train'][LM_COLUMNS].to_numpy()))
validation_metrics = classification_metrics(splits['validation'].class_id, model.predict_proba(splits['validation'][LM_COLUMNS].to_numpy()))
display(pd.DataFrame([train_metrics,validation_metrics],index=['train','validation'])[['accuracy','macro_f1']])
pd.DataFrame([{'smoke_test':SMOKE_TEST,'train_accuracy':train_metrics['accuracy'],'validation_accuracy':validation_metrics['accuracy'],
    'train_macro_f1':train_metrics['macro_f1'],'validation_macro_f1':validation_metrics['macro_f1']}]).to_csv(output_dir/'history.csv',index=False)
joblib.dump(model, output_dir/'model.joblib')
save_json(output_dir/'model_config.json', model_config)
# Cấu hình cố định; không refit train+validation.
test_probabilities = model.predict_proba(splits['test'][LM_COLUMNS].to_numpy())

### Đánh giá test đã cố định
Accuracy, macro-Precision/Recall/F1 và báo cáo từng lớp. Gallery chỉ phục vụ phân tích lỗi.

In [ ]:
metrics, predictions = export_evaluation(
    output_dir, MODEL_NAME, splits['test'], test_probabilities, protocol, env,
    training_time, fit_wall_time, validation_metrics, model_config)
show_errors(predictions, root, output_dir/'misclassified_gallery.png',smoke=SMOKE_TEST)
write_model_report(output_dir, metrics, predictions)

### Cách đọc kết quả và sử dụng model
Confusion matrix hàng là nhãn thật, cột là nhãn dự đoán; tỷ lệ chuẩn hóa theo hàng cho thấy
lớp nào bị nhầm nhiều. Quan sát thêm cặp `four`–`palm`, macro-F1 và gallery lỗi.
Test chỉ được dùng sau khi mô hình/checkpoint đã cố định. Không thay cấu hình dựa trên test.

Lưu **Save Version → Save & Run All** để giữ artifact rồi gắn output vào notebook 04.
`model_config.json` ghi kiến trúc và preprocessing; `class_mapping.json` ghi nhãn.
RF dùng `joblib.load`; MLP/CNN khởi tạo đúng lớp trong notebook rồi tải `state_dict`
bằng `torch.load(..., map_location='cpu', weights_only=True)`, đặt `eval()`.
MLP phải tải `scaler.joblib`; CNN suy luận bằng `image_tensor(image, training=False)`.
Thời gian huấn luyện có kèm thiết bị; chưa đo FPS hay MediaPipe/crop/đọc ảnh end-to-end.